# Notebook to save uncertainties for before/after quadrature change

In [1]:
import rmgpy.tools.uncertainty
import rmgpy.chemkin
import numpy as np
import pickle

In [2]:
# load the GAV mechanism
chemkin = 'thermo_chemkin_files/GAV.inp'
species_dict = 'thermo_chemkin_files/GAV_species_dictionary.txt'

uncertainty = rmgpy.tools.uncertainty.Uncertainty()
uncertainty.load_model(chemkin, species_dict)



In [3]:
uncertainty.database = rmgpy.data.rmg.RMGDatabase()
uncertainty.database.load(
    path=rmgpy.settings['database.directory'],
    thermo_libraries=[],
    transport_libraries=[],
    reaction_libraries=[],
    seed_mechanisms=[],
    kinetics_families=[],
    kinetics_depositories=[],
    depository=False,
)

In [4]:
uncertainty.extract_sources_from_model()
uncertainty.assign_parameter_uncertainties()

In [ ]:
# np.save('thermo_chemkin_files/original_GAV_input_uncertainties.npy', uncertainty.thermo_input_uncertainties)

# # also save the sources dictionary
# with open('thermo_chemkin_files/original_GAV_sources.pickle', 'wb') as f:
#     pickle.dump(uncertainty.species_sources_dict, f)


In [5]:
np.save('thermo_chemkin_files/quadrature_GAV_input_uncertainties.npy', uncertainty.thermo_input_uncertainties)

# also save the sources dictionary
with open('thermo_chemkin_files/quadrature_GAV_sources.pickle', 'wb') as f:
    pickle.dump(uncertainty.species_sources_dict, f)


In [ ]:
# read in the sources pickle
with open('thermo_chemkin_files/original_GAV_sources.pickle', 'rb') as f:
    data = pickle.load(f)


In [40]:
data

{Species(index=0, label="H2O2", thermo=NASA(polynomials=[NASAPolynomial(coeffs=[3.00676,0.00886959,-5.12544e-06,4.40872e-10,4.56944e-13,-17650.7,8.45904], Tmin=(298,'K'), Tmax=(1165.97,'K')), NASAPolynomial(coeffs=[5.60975,0.00306723,-1.6843e-06,7.73825e-10,-1.07699e-13,-18470.2,-5.41027], Tmin=(1165.97,'K'), Tmax=(3000,'K'))], Tmin=(298,'K'), Tmax=(3000,'K'), comment="""Thermo group additivity estimation: group(O2s-OsH) + group(O2s-OsH)"""), molecule=[Molecule(smiles="OO")], molecular_weight=(34.0147,'amu')): {'GAV': {'group': [(<Entry index=2317 label="O2s-OsH">,
     2)]}},
 Species(index=1, label="HO", thermo=NASA(polynomials=[NASAPolynomial(coeffs=[3.77865,-0.000694119,1.39704e-07,9.12493e-10,-4.46734e-13,3590.45,0.762518], Tmin=(298,'K'), Tmax=(1053.39,'K')), NASAPolynomial(coeffs=[3.38268,-0.000125874,6.62462e-07,-2.61288e-10,3.18914e-14,3725.77,2.93977], Tmin=(1053.39,'K'), Tmax=(3000,'K'))], Tmin=(298,'K'), Tmax=(3000,'K'), comment="""Thermo group additivity estimation: group(

In [23]:
for i in range(len(uncertainty.species_list)):
    try:
        source = uncertainty.database.thermo.extract_source_from_comments(uncertainty.species_list[i])
    except ValueError:
        print(i, uncertainty.species_list[i].smiles, 'no source found')
    

93 [N-]=[NH2+] no source found


In [21]:
uncertainty.species_list[93].thermo

NASA(polynomials=[NASAPolynomial(coeffs=[1.36185,-0.0100087,2.53024e-05,-2.63564e-08,9.67916e-12,-137.208,-6.37919], Tmin=(298,'K'), Tmax=(875.73,'K')), NASAPolynomial(coeffs=[0.723842,-0.00130092,4.635e-07,1.0739e-10,-3.22987e-14,-247.621,-4.65426], Tmin=(875.73,'K'), Tmax=(3000,'K'))], Tmin=(298,'K'), Tmax=(3000,'K'), comment="""Thermo group additivity estimation: missing(N5dc-H0H0N1dc) + missing(N1dc-N5dc)""")

In [ ]:
uncertainty.load_database(
    thermo_libraries=[],
    kinetics_families=[],
    reaction_libraries=[],
)

AttributeError: 'NoneType' object has no attribute 'startswith'

In [22]:
database = rmgpy.data.rmg.RMGDatabase()
database.load(
    path = rmgpy.settings['database.directory'],
    thermo_libraries = thermo_libraries,
    transport_libraries = [],
    reaction_libraries = [],
    seed_mechanisms = [],
    kinetics_families = [],
    kinetics_depositories = [],
    depository = False,
)

In [4]:
def get_i_thing(thing, thing_list):
    for i in range(len(thing_list)):
        if thing.is_isomorphic(thing_list[i]):
            return i
    return -1

def species_in_list(sp, sp_list):
    return get_i_thing(sp, sp_list) != -1
    

In [23]:
skip_list = [
    rmgpy.species.Species(smiles='[Ar]'),  # GAV fails for this
    rmgpy.species.Species(smiles='[He]'),  # GAV fails for this
    rmgpy.species.Species(smiles='[Ne]'),  # GAV fails for this
    rmgpy.species.Species(smiles='[H][H]'),  # GAV produces no comment for this
    rmgpy.species.Species(smiles='[C-]=[N+]=O'),  # GAV fails for this
    rmgpy.species.Species(smiles='[C]=N'),  # GAV fails on this
]